# Ireland Electricity Price Analytics

**Project 02 — Ireland / Energy / Data**

This notebook analyses how Irish business electricity prices changed from 2015–2025, places that decade inside a longer 2008–2025 history, compares electricity with business gas prices, examines differences across business consumption bands, and prepares the dataset for later demand × price and forecasting work.

### Core research question

> **Why did Irish business electricity prices change so sharply, who paid what, and which variables should we carry into a defensible 2026–2035 forecasting model?**

**Publication phase:** historical analysis, market-driver testing, model validation and 2026–2035 scenario modelling are complete.  
**Forecasting approach:** the project reports transparent scenarios rather than presenting one long-run point estimate as certain.


## Source hierarchy

Primary sources used in this version:

1. **SEAI Energy Price Trends** — weighted-average business electricity and gas prices.
2. **SEAI EPR Semester Update, July–December 2025** — 2025 S2 weighted average, business consumption-band prices, EU ranking and market shares.
3. **SEAI Commercial Fuel Cost Comparison** — supporting quarterly fuel-cost context.

Official links:
- https://www.seai.ie/data-and-insights/seai-statistics/prices
- https://www.seai.ie/data-and-insights/seai-statistics/energy-prices
- https://www.seai.ie/sites/default/files/publications/EPR-Electricity-Gas-Prices.pdf

**Important:** the commercial fuel-cost table is not treated as 72 independent electricity-price observations because electricity/gas values repeat across many adjacent quarters. The biannual EPR series is our main target-price series.


## Publication summary

Key findings from the completed analysis:

- Weighted-average Irish business electricity prices rose from **13.79 c/kWh in 2015** to **23.49 c/kWh in 2025** on an annual-average basis.
- The semester peak was **28.40 c/kWh in 2022 S2**, during the European energy-price shock.
- In 2025 S2, the largest public non-household consumption band (**IG, >150,000 MWh/year**) paid **15.9 c/kWh**, compared with a **22.7 c/kWh** weighted business average.
- On 2018–2025 walk-forward validation, a simple **last-price naive model** beat Holt, trend and gas-augmented alternatives.
- Adding lagged wholesale features improved explanation but **did not beat the naive baseline out of sample**.
- The long-run output is therefore a **scenario envelope**, not a single deterministic price forecast. For weighted-average business prices in real 2025 money, the 2035 scenarios are approximately **21.2 / 24.4 / 30.1 c/kWh** (low / central / high).

These scenarios are analytical sensitivities, not official forecasts or statistical confidence intervals.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_NAME = "Ireland Electricity Price Analytics"

# Colab / Google Drive
if Path("/content").exists():
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
    except Exception:
        PROJECT_DIR = Path.cwd()
else:
    # Local/GitHub fallback
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_DIR = next((p for p in candidates if (p / "data").exists()), Path.cwd())

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
CHARTS_DIR = PROJECT_DIR / "charts"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR)


In [ ]:
electricity = pd.read_csv(RAW_DIR / "seai_business_electricity_biannual_2008_2025.csv")
gas = pd.read_csv(RAW_DIR / "seai_business_gas_biannual_2008_2025.csv")
quarterly_fuels = pd.read_csv(RAW_DIR / "seai_commercial_fuel_cost_quarterly_2008_2025.csv")
bands_2025s2 = pd.read_csv(RAW_DIR / "seai_business_electricity_bands_2025S2.csv")

display(electricity.head())
display(bands_2025s2)


## 1. Validate the source snapshots

Before analysing prices, check period uniqueness, numeric coverage, expected time ranges and missing values.


In [ ]:
def validate_period_table(df, period_col):
    print("Rows:", len(df))
    print("Unique periods:", df[period_col].nunique())
    print("Duplicate periods:", int(df[period_col].duplicated().sum()))
    print("First period:", df[period_col].iloc[0])
    print("Last period:", df[period_col].iloc[-1])
    print("\nMissing values:")
    display(df.isna().sum().to_frame("missing"))

print("BUSINESS ELECTRICITY")
validate_period_table(electricity, "period")

print("\nBUSINESS GAS")
validate_period_table(gas, "period")

assert electricity["period"].is_unique
assert gas["period"].is_unique


### Data-quality note: “quarterly” does not mean quarterly-independent

The SEAI commercial fuel comparison is published as a quarterly series, but business electricity and gas values often repeat across adjacent quarters. That makes it useful as contextual fuel-cost history, but not a good excuse to artificially inflate the target-price sample size.

The direct EPR business-price series is collected and reported **biannually**, so the forecasting target will preserve that cadence unless we later obtain a genuinely higher-frequency market-price target.


In [ ]:
repeat_rate_electricity = (
    quarterly_fuels["electricity_avg_commercial_c_per_kwh"]
    .diff()
    .eq(0)
    .mean()
)
repeat_rate_gas = (
    quarterly_fuels["gas_avg_commercial_c_per_kwh"]
    .diff()
    .eq(0)
    .mean()
)

print(f"Adjacent-quarter repeats — electricity: {repeat_rate_electricity:.1%}")
print(f"Adjacent-quarter repeats — gas: {repeat_rate_gas:.1%}")


## 2. Build the core biannual business-price dataset

We combine Irish weighted-average business electricity and gas prices at the same semester frequency.

Prices are in **euro cent/kWh, ex VAT**.


In [ ]:
prices = (
    electricity[["period", "ireland_c_per_kwh", "euro_area_c_per_kwh", "eu27_c_per_kwh"]]
    .rename(columns={
        "ireland_c_per_kwh": "electricity_ie_c_per_kwh",
        "euro_area_c_per_kwh": "electricity_euro_area_c_per_kwh",
        "eu27_c_per_kwh": "electricity_eu27_c_per_kwh",
    })
    .merge(
        gas[["period", "ireland_c_per_kwh"]]
        .rename(columns={"ireland_c_per_kwh": "gas_ie_c_per_kwh"}),
        on="period",
        how="left",
        validate="one_to_one",
    )
)

prices["year"] = prices["period"].str[:4].astype(int)
prices["semester"] = prices["period"].str[-1].astype(int)
prices["electricity_yoy_pct"] = prices["electricity_ie_c_per_kwh"].pct_change(2) * 100
prices["gas_yoy_pct"] = prices["gas_ie_c_per_kwh"].pct_change(2) * 100

prices.to_csv(PROCESSED_DIR / "business_energy_prices_biannual_2008_2025.csv", index=False)
display(prices.tail(10))


## 3. The 2015–2025 story window

The public-facing story will align with Project 01's decade. The modelling history remains longer.


In [ ]:
story = prices.query("2015 <= year <= 2025").copy()

annual = (
    story.groupby("year", as_index=False)
    .agg(
        business_electricity_c_per_kwh=("electricity_ie_c_per_kwh", "mean"),
        business_gas_c_per_kwh=("gas_ie_c_per_kwh", "mean"),
    )
)

annual["electricity_yoy_pct"] = annual["business_electricity_c_per_kwh"].pct_change() * 100
annual["gas_yoy_pct"] = annual["business_gas_c_per_kwh"].pct_change() * 100
annual.to_csv(PROCESSED_DIR / "business_energy_prices_annual_2015_2025.csv", index=False)

p2015 = annual.loc[annual["year"].eq(2015), "business_electricity_c_per_kwh"].iloc[0]
p2025 = annual.loc[annual["year"].eq(2025), "business_electricity_c_per_kwh"].iloc[0]
peak = prices.loc[prices["electricity_ie_c_per_kwh"].idxmax()]

print(f"2015 annual average: {p2015:.2f} c/kWh")
print(f"2025 annual average: {p2025:.2f} c/kWh")
print(f"2015→2025 change: {(p2025/p2015-1)*100:.1f}%")
print(f"Peak semester: {peak['period']} at {peak['electricity_ie_c_per_kwh']:.1f} c/kWh")
print(f"2025 S2: {prices.iloc[-1]['electricity_ie_c_per_kwh']:.1f} c/kWh")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(prices["period"], prices["electricity_ie_c_per_kwh"], marker="o", linewidth=2)
ax.set_title("Ireland business electricity price, 2008–2025")
ax.set_ylabel("Euro cent/kWh (ex VAT)")
ax.set_xlabel("Semester")
ax.tick_params(axis="x", rotation=70)
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "01_business_electricity_price_2008_2025.png", bbox_inches="tight", dpi=180)
plt.show()


### First finding

The decade is **not** a smooth upward trend.

Business electricity prices were relatively stable through 2020, rose sharply during the 2021–2022 European energy shock, peaked at **28.4 c/kWh in 2022 S2**, then partially retreated. The 2025 S2 weighted average was **22.7 c/kWh** — about 20% below the peak, but still materially above the pre-crisis level.


## 4. Electricity and gas moved together — especially during the crisis

This is an important feature-discovery step, not a causal conclusion.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(prices["period"], prices["electricity_ie_c_per_kwh"], marker="o", linewidth=2, label="Business electricity")
ax.plot(prices["period"], prices["gas_ie_c_per_kwh"], marker="o", linewidth=2, label="Business gas")
ax.set_title("Ireland business electricity and gas prices")
ax.set_ylabel("Euro cent/kWh (ex VAT)")
ax.set_xlabel("Semester")
ax.tick_params(axis="x", rotation=70)
ax.grid(True, axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(CHARTS_DIR / "02_business_electricity_vs_gas_2008_2025.png", bbox_inches="tight", dpi=180)
plt.show()

overall_corr = prices["electricity_ie_c_per_kwh"].corr(prices["gas_ie_c_per_kwh"])
pre_crisis_corr = (
    prices.query("year <= 2020")["electricity_ie_c_per_kwh"]
    .corr(prices.query("year <= 2020")["gas_ie_c_per_kwh"])
)
crisis_corr = (
    prices.query("year >= 2021")["electricity_ie_c_per_kwh"]
    .corr(prices.query("year >= 2021")["gas_ie_c_per_kwh"])
)

print(f"Correlation, 2008–2025: {overall_corr:.3f}")
print(f"Correlation, 2008–2020: {pre_crisis_corr:.3f}")
print(f"Correlation, 2021–2025: {crisis_corr:.3f}")


The overall correlation is high, but most of that relationship is strengthened by the common 2021–2023 price shock. We will **not** interpret this as proof that gas alone determines electricity prices.

Later phases will add:
- SEMOpx wholesale electricity prices
- gas-market / wholesale fuel variables
- electricity demand and data-centre demand
- renewable generation
- net imports/interconnection
- CRU network tariff / network-cost variables
- taxes and levies where comparable


## 5. Ireland compared with the Euro area and EU-27

The SEAI series based on Eurostat lets us compare the weighted average business price across markets.

The 2025 S2 SEAI report ranks Ireland's weighted-average business electricity price **2nd in the EU**. The directly comparable EU-average series in this source snapshot currently runs through 2025 S1.


In [ ]:
eu_compare = prices.dropna(subset=["electricity_euro_area_c_per_kwh", "electricity_eu27_c_per_kwh"]).copy()

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(eu_compare["period"], eu_compare["electricity_ie_c_per_kwh"], marker="o", linewidth=2, label="Ireland")
ax.plot(eu_compare["period"], eu_compare["electricity_euro_area_c_per_kwh"], linewidth=2, label="Euro area")
ax.plot(eu_compare["period"], eu_compare["electricity_eu27_c_per_kwh"], linewidth=2, label="EU-27")
ax.set_title("Business electricity price: Ireland vs Euro area and EU-27")
ax.set_ylabel("Euro cent/kWh")
ax.set_xlabel("Semester")
ax.tick_params(axis="x", rotation=70)
ax.grid(True, axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(CHARTS_DIR / "04_ireland_vs_eu_business_electricity.png", bbox_inches="tight", dpi=180)
plt.show()

s1_2025 = eu_compare.loc[eu_compare["period"].eq("2025S1")].iloc[0]
print(f"Ireland 2025 S1: {s1_2025['electricity_ie_c_per_kwh']:.2f} c/kWh")
print(f"Euro area 2025 S1: {s1_2025['electricity_euro_area_c_per_kwh']:.2f} c/kWh")
print(f"EU-27 2025 S1: {s1_2025['electricity_eu27_c_per_kwh']:.2f} c/kWh")


## 6. Scale matters: 2025 S2 prices by business consumption band

SEAI reports a clear unit-price gradient: larger electricity users pay less per kWh on average.

This is especially relevant to Project 03, but **Band IG is not a “data-centre tariff.”** It is a public benchmark for all business consumers using more than 150,000 MWh/year.


In [ ]:
display(bands_2025s2[[
    "band", "annual_consumption", "price_c_per_kwh",
    "change_12m_pct", "eu_rank", "market_share_pct"
]])

weighted_2025s2 = 22.7
ig = bands_2025s2.loc[bands_2025s2["band"].eq("IG"), "price_c_per_kwh"].iloc[0]
ia = bands_2025s2.loc[bands_2025s2["band"].eq("IA"), "price_c_per_kwh"].iloc[0]

print(f"Weighted average: {weighted_2025s2:.1f} c/kWh")
print(f"Band IG: {ig:.1f} c/kWh ({(ig/weighted_2025s2-1)*100:.1f}% vs weighted average)")
print(f"Band IA: {ia:.1f} c/kWh")
print(f"IG vs IA unit-price difference: {(ig/ia-1)*100:.1f}%")

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.bar(bands_2025s2["band"], bands_2025s2["price_c_per_kwh"])
ax.axhline(weighted_2025s2, linestyle="--", linewidth=1.5, label="Weighted average")
ax.set_title("Ireland business electricity price by consumption band — 2025 S2")
ax.set_ylabel("Euro cent/kWh (ex VAT)")
ax.set_xlabel("Business consumption band")
ax.grid(True, axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(CHARTS_DIR / "03_business_electricity_by_band_2025S2.png", bbox_inches="tight", dpi=180)
plt.show()


## 7. What we can say already

From the first official datasets:

- The 2015–2025 business electricity annual average increased by roughly **70%**.
- The decade contains a major structural break around the 2021–2022 energy crisis; a simple straight-line forecast would be inappropriate.
- Electricity and business gas prices are strongly correlated over the full sample, but the relationship becomes much stronger during/after 2021.
- Ireland's business electricity price remains high in European comparison.
- Very large users pay substantially less per kWh than small users, so the later Demand × Price project must not apply an average business tariff directly to data-centre consumption.

These findings determine the next data-engineering step.


## 8. Next research layer before forecasting

We will now add variables that can plausibly explain price movement:

1. **SEMOpx wholesale electricity price** — preferably monthly or daily aggregated to semester/quarter.
2. **CRU network tariff / allowed revenue series** — network-cost component.
3. **SEAI / CSO electricity demand and generation mix** — demand, wind, gas generation and imports.
4. **Project 01 data-centre demand** — joined only after the price model is independently understood.
5. **Eurostat price components** — energy/supply, networks, taxes/levies where comparable.

Only then will we compare:
- naive baseline
- Holt / ETS
- ARIMA/SARIMA
- regression with exogenous variables
- regularised regression / tree model only if sample size supports it

Validation will use **walk-forward / rolling-origin evaluation**, never a random split.


## 9. Wholesale electricity layer

Retail electricity prices do not move independently of the wholesale market.

For this project we use two complementary official measures:

- **CSO WPM37** — monthly Wholesale Price Index for Electricity. This becomes our higher-frequency modelling feature.
- **SEMOpx Day-Ahead Market (DAM)** — annual average clearing-price benchmarks in €/MWh. This gives the wholesale series an interpretable absolute-price scale.

CSO states that from January 2019 onward its electricity index uses the Day-Ahead Single Market Price. The 2021=100 rebasing changes the scale, not the underlying month-to-month price movement.

Sources:
- CSO WPM37 dataset: https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/WPM37/CSV/1.0/en
- CSO methodology: https://www.cso.ie/en/methods/prices/wholesalepriceindex/introductionofupdatedppiwpiseriesbaseyear2021100/chapter3examplesofchangestoseries/
- SEMOpx historical-data guide: https://www.semopx.com/market-data/static-reports/semopx-historical-market-data-final-v2.pdf


In [ ]:
semopx_annual = pd.read_csv(RAW_DIR / "semopx_dam_annual_2021_2025.csv")
semopx_annual["avg_dam_c_per_kwh"] = semopx_annual["avg_dam_eur_mwh"] / 10

display(semopx_annual[["year", "avg_dam_eur_mwh", "avg_dam_c_per_kwh"]])

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(
    semopx_annual["year"],
    semopx_annual["avg_dam_eur_mwh"],
    marker="o",
    linewidth=2.5,
)
ax.set_title("SEMOpx annual average Day-Ahead Market price")
ax.set_ylabel("€/MWh")
ax.set_xlabel("Year")
ax.grid(axis="y", alpha=0.25)
plt.show()


### The 2022 wholesale shock is obvious

The SEMOpx annual DAM average rose from **€135.52/MWh in 2021** to **€225.57/MWh in 2022**, then fell sharply in 2023–2024.

This is already different from the retail series: business retail prices remained elevated after wholesale prices fell, which is consistent with the idea that retail prices can adjust with lags and include costs beyond the spot wholesale market.


In [ ]:
retail_vs_dam = (
    annual[["year", "business_electricity_c_per_kwh"]]
    .merge(
        semopx_annual[["year", "avg_dam_eur_mwh", "avg_dam_c_per_kwh"]],
        on="year",
        how="inner",
    )
)

retail_vs_dam["retail_minus_dam_c_per_kwh"] = (
    retail_vs_dam["business_electricity_c_per_kwh"]
    - retail_vs_dam["avg_dam_c_per_kwh"]
)

display(retail_vs_dam)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(
    retail_vs_dam["year"],
    retail_vs_dam["business_electricity_c_per_kwh"],
    marker="o",
    linewidth=2.5,
    label="Business retail weighted average",
)
ax.plot(
    retail_vs_dam["year"],
    retail_vs_dam["avg_dam_c_per_kwh"],
    marker="o",
    linewidth=2.5,
    label="SEMOpx DAM average",
)
ax.set_title("Business retail price vs Day-Ahead Market price")
ax.set_ylabel("cent/kWh")
ax.set_xlabel("Year")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.show()


> **Do not interpret the retail-minus-DAM difference as a supplier margin.**
>
> The retail price includes other cost layers and procurement effects, including networks, hedging, taxes/levies and supplier costs. The DAM series is an ex-ante wholesale clearing-price benchmark.


## 10. Download the official monthly CSO wholesale-electricity index

This cell downloads the complete **WPM37** table directly from the CSO PXStat API and saves the raw snapshot in the project.

The parser is deliberately tolerant of PXStat column-name changes: it detects the month, energy type and value fields rather than assuming a fixed column order.


In [ ]:
import requests
import re

CSO_WPM37_URL = (
    "https://ws.cso.ie/public/api.restful/"
    "PxStat.Data.Cube_API.ReadDataset/WPM37/CSV/1.0/en"
)
WPM37_RAW = RAW_DIR / "cso_wpm37_wholesale_energy.csv"

if not WPM37_RAW.exists():
    response = requests.get(CSO_WPM37_URL, timeout=60)
    response.raise_for_status()
    WPM37_RAW.write_bytes(response.content)
    print("Downloaded:", WPM37_RAW)
else:
    print("Using existing raw snapshot:", WPM37_RAW)

wpm37 = pd.read_csv(WPM37_RAW)
print("Rows:", len(wpm37))
print("Columns:", list(wpm37.columns))
display(wpm37.head())


In [ ]:
def find_col(columns, include_any, exclude_any=()):
    for col in columns:
        low = str(col).lower()
        if any(token in low for token in include_any) and not any(token in low for token in exclude_any):
            return col
    return None

month_col = find_col(wpm37.columns, ["month"])
energy_col = find_col(wpm37.columns, ["type of energy", "energy"])
value_col = next((c for c in wpm37.columns if str(c).strip().upper() == "VALUE"), None)

if month_col is None:
    # PXStat files often include a TLIST(M1) monthly code.
    month_col = find_col(wpm37.columns, ["tlist"])
if energy_col is None or value_col is None or month_col is None:
    raise ValueError(
        f"Could not identify required WPM37 columns. "
        f"month={month_col}, energy={energy_col}, value={value_col}"
    )

wholesale = wpm37[
    wpm37[energy_col].astype(str).str.contains("electricity", case=False, na=False)
].copy()

# Prefer the human-readable month label; if it is a numeric code, handle YYYYMM.
month_text = wholesale[month_col].astype(str).str.strip()
parsed_date = pd.to_datetime(month_text, errors="coerce")

if parsed_date.isna().mean() > 0.5:
    digits = month_text.str.replace(r"\D", "", regex=True)
    parsed_date = pd.to_datetime(digits.str[:6], format="%Y%m", errors="coerce")

wholesale["date"] = parsed_date
wholesale["wholesale_electricity_index"] = pd.to_numeric(
    wholesale[value_col], errors="coerce"
)

wholesale = (
    wholesale[["date", "wholesale_electricity_index"]]
    .dropna()
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

display(wholesale.head())
display(wholesale.tail())

wholesale.to_csv(
    PROCESSED_DIR / "cso_wholesale_electricity_monthly.csv",
    index=False,
)


In [ ]:
story_wholesale = wholesale[
    wholesale["date"].between("2015-01-01", "2025-12-31")
].copy()

fig, ax = plt.subplots(figsize=(12, 5.8))
ax.plot(
    story_wholesale["date"],
    story_wholesale["wholesale_electricity_index"],
    linewidth=2.0,
)
ax.axhline(100, linewidth=1, linestyle="--", alpha=0.6)
ax.set_title("Ireland wholesale electricity price index, 2015–2025")
ax.set_ylabel("Index (2021 average = 100)")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
plt.show()


## 11. Does wholesale movement lead retail prices?

Retail prices are observed every six months, so we aggregate the monthly wholesale index to the same semester frequency.

Then we test:

- **Lag 0:** wholesale and retail in the same semester
- **Lag 1:** wholesale leads retail by one semester
- **Lag 2:** wholesale leads retail by one year
- up to **Lag 4**

These are relationship diagnostics, not causal estimates.


In [ ]:
wholesale_half = wholesale.copy()
wholesale_half["year"] = wholesale_half["date"].dt.year
wholesale_half["semester"] = np.where(
    wholesale_half["date"].dt.month <= 6, "S1", "S2"
)
wholesale_half["period"] = (
    wholesale_half["year"].astype(str) + wholesale_half["semester"]
)

wholesale_half = (
    wholesale_half.groupby(["year", "semester", "period"], as_index=False)
    ["wholesale_electricity_index"]
    .mean()
)

retail_half = prices[
    ["period", "year", "semester", "electricity_ie_c_per_kwh"]
].copy()

wholesale_retail = (
    retail_half.merge(
        wholesale_half[["period", "wholesale_electricity_index"]],
        on="period",
        how="inner",
    )
    .sort_values(["year", "semester"])
    .reset_index(drop=True)
)

# Normalise both to their 2021 annual average = 100 for shape comparison.
base_2021 = wholesale_retail.query("year == 2021")
retail_base = base_2021["electricity_ie_c_per_kwh"].mean()
wholesale_base = base_2021["wholesale_electricity_index"].mean()

wholesale_retail["retail_index_2021_100"] = (
    wholesale_retail["electricity_ie_c_per_kwh"] / retail_base * 100
)
wholesale_retail["wholesale_index_2021_100"] = (
    wholesale_retail["wholesale_electricity_index"] / wholesale_base * 100
)

display(wholesale_retail.tail(12))

wholesale_retail.to_csv(
    PROCESSED_DIR / "wholesale_retail_halfyear.csv",
    index=False,
)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.8))
ax.plot(
    wholesale_retail["period"],
    wholesale_retail["retail_index_2021_100"],
    marker="o",
    label="Business retail price",
)
ax.plot(
    wholesale_retail["period"],
    wholesale_retail["wholesale_index_2021_100"],
    marker="o",
    label="Wholesale electricity",
)
ax.axhline(100, linewidth=1, linestyle="--", alpha=0.6)
ax.set_title("Retail and wholesale electricity prices — normalised comparison")
ax.set_ylabel("Index (2021 average = 100)")
ax.tick_params(axis="x", rotation=60)
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.show()


In [ ]:
lag_results = []

for lag in range(5):
    temp = wholesale_retail[
        ["electricity_ie_c_per_kwh", "wholesale_electricity_index"]
    ].copy()
    temp["wholesale_lead"] = temp["wholesale_electricity_index"].shift(lag)
    valid = temp.dropna()
    lag_results.append({
        "wholesale_lead_semesters": lag,
        "observations": len(valid),
        "correlation": valid["electricity_ie_c_per_kwh"].corr(
            valid["wholesale_lead"]
        ),
    })

lag_results = pd.DataFrame(lag_results)
display(lag_results)

lag_results.to_csv(
    PROCESSED_DIR / "wholesale_retail_lag_correlations.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    lag_results["wholesale_lead_semesters"].astype(str),
    lag_results["correlation"],
)
ax.set_title("Wholesale-leading-retail correlation by lag")
ax.set_xlabel("Wholesale lead (semesters)")
ax.set_ylabel("Pearson correlation")
ax.axhline(0, linewidth=1)
ax.grid(axis="y", alpha=0.25)
plt.show()


### How to read the lag test

The strongest lag is a **candidate feature relationship**, not proof that wholesale prices cause the observed retail movement by exactly that delay.

Supplier hedging, contract structures, network tariffs, taxes/levies and other market effects can all alter the timing and magnitude of retail-price changes.

The next model-building stage will compare predictive performance with and without lagged wholesale features using walk-forward validation.


## 12. Optional SEMOpx historical workbook download

SEMOpx provides two official Ex-Ante Market look-back workbooks covering the market from go-live.

These files are useful for validating absolute DAM prices and, if required, building a second monthly wholesale series directly from SEMOpx rather than relying only on the CSO index.


In [ ]:
SEMOPX_LOOKBACK_URLS = {
    "go_live_to_2020": (
        "https://www.semopx.com/documents/general-publications/lookback_mkt.xlsx"
    ),
    "2021_onwards": (
        "https://www.semopx.com/documents/general-publications/Lookback2_mkt.xlsx"
    ),
}

for label, url in SEMOPX_LOOKBACK_URLS.items():
    target = RAW_DIR / f"semopx_{label}.xlsx"
    if not target.exists():
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        target.write_bytes(response.content)
        print("Downloaded", target.name, f"({target.stat().st_size / 1e6:.1f} MB)")
    else:
        print("Using existing", target.name)

for label in SEMOPX_LOOKBACK_URLS:
    path = RAW_DIR / f"semopx_{label}.xlsx"
    book = pd.ExcelFile(path)
    print("\n", path.name, "sheets:", book.sheet_names)
    for sheet in book.sheet_names[:10]:
        sample = pd.read_excel(path, sheet_name=sheet, header=None, nrows=12)
        print(" -", sheet, sample.shape)
        # Show only sheets whose first rows mention DAM / price / day-ahead.
        text_blob = " ".join(sample.astype(str).fillna("").to_numpy().ravel()).lower()
        if any(token in text_blob for token in ["dam", "day ahead", "price"]):
            display(sample)


## Wholesale layer status

After these cells run successfully we will have:

- an official monthly CSO wholesale-electricity signal;
- SEMOpx annual absolute DAM benchmarks;
- matched half-year retail and wholesale series;
- lag diagnostics suitable for feature selection;
- the original SEMOpx look-back workbooks saved in `data/raw/`.

The next research layer is **network costs / tariffs and policy components**, followed by a first walk-forward forecasting benchmark.


## 13. Price components and network context

A retail electricity price is not just the wholesale energy price.

SEAI describes the non-household weighted-average bill as a combination of:

- **Energy and Supply**
- **Network Costs**
- **VAT**
- **Renewable Taxes**
- **Capacity Taxes**
- **Environmental Taxes**
- **Other**

Eurostat publishes the same component framework for non-household consumption bands in dataset `nrg_pc_205_c`, with annual coverage from 2017 onward.

For our modelling, components are primarily **explanatory and scenario variables**. We should not insert an annual network-cost series into a half-yearly forecast unless its timing and unit are aligned correctly.


In [ ]:
# Install the small Eurostat helper package only if this Colab runtime does not have it.
try:
    import eurostat
except ImportError:
    %pip -q install eurostat
    import eurostat

EUROSTAT_COMPONENT_CODE = "nrg_pc_205_c"

# Pull all Irish non-household electricity price components in EUR.
component_raw = eurostat.get_data_df(
    EUROSTAT_COMPONENT_CODE,
    filter_pars={
        "geo": "IE",
        "currency": "EUR",
        "startPeriod": 2017,
        "endPeriod": 2025,
    },
)

display(component_raw.head())
print("Rows:", len(component_raw))
print("Parameters:", eurostat.get_pars(EUROSTAT_COMPONENT_CODE))

# Preserve the exact download used by this run.
component_raw.to_csv(
    RAW_DIR / "eurostat_nrg_pc_205_c_ireland_2017_2025.csv",
    index=False,
)


In [ ]:
# Convert the wide Eurostat table to long form and attach readable component labels.
id_cols = [c for c in component_raw.columns if not str(c).isdigit()]
year_cols = [c for c in component_raw.columns if str(c).isdigit()]

components = component_raw.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="year",
    value_name="eur_per_kwh",
)

components["year"] = pd.to_numeric(components["year"], errors="coerce")
components["eur_per_kwh"] = pd.to_numeric(components["eur_per_kwh"], errors="coerce")
components["c_per_kwh"] = components["eur_per_kwh"] * 100

# Find the code columns robustly.
nrg_prc_col = next(c for c in components.columns if str(c).lower() == "nrg_prc")
nrg_cons_col = next(c for c in components.columns if str(c).lower() == "nrg_cons")

component_dic = dict(eurostat.get_dic(EUROSTAT_COMPONENT_CODE, "nrg_prc"))
band_dic = dict(eurostat.get_dic(EUROSTAT_COMPONENT_CODE, "nrg_cons"))

components["component"] = components[nrg_prc_col].map(component_dic)
components["band_description"] = components[nrg_cons_col].map(band_dic)

components = components.dropna(subset=["year", "c_per_kwh"]).copy()
components.to_csv(
    PROCESSED_DIR / "eurostat_business_price_components_2017_2025.csv",
    index=False,
)

display(
    components[
        [nrg_cons_col, "band_description", "component", "year", "c_per_kwh"]
    ].tail(20)
)


In [ ]:
# Large-user benchmark: Band IG (>=150,000 MWh/year), where available.
ig_code = "MWH_GE150000"
ig_components = components[components[nrg_cons_col].eq(ig_code)].copy()

wanted = [
    "Energy and supply",
    "Network costs",
    "Value added tax (VAT)",
    "Renewable taxes",
    "Capacity taxes",
    "Environmental taxes",
    "Other",
]

# Flexible text matching because Eurostat labels can change punctuation/case.
def component_family(label):
    low = str(label).lower()
    if "energy and supply" in low:
        return "Energy and supply"
    if "network" in low:
        return "Network costs"
    if "value added tax" in low or low == "vat":
        return "VAT"
    if "renewable" in low:
        return "Renewable taxes"
    if "capacity" in low:
        return "Capacity taxes"
    if "environmental" in low:
        return "Environmental taxes"
    if low.strip() == "other" or "other" in low:
        return "Other"
    return None

ig_components["component_family"] = ig_components["component"].map(component_family)
ig_plot = (
    ig_components.dropna(subset=["component_family"])
    .pivot_table(
        index="year",
        columns="component_family",
        values="c_per_kwh",
        aggfunc="first",
    )
    .sort_index()
)

display(ig_plot)

if not ig_plot.empty:
    ax = ig_plot.plot(kind="bar", stacked=True, figsize=(12, 6))
    ax.set_title("Band IG electricity price components — Ireland")
    ax.set_ylabel("cent/kWh")
    ax.set_xlabel("Year")
    ax.legend(title="")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("Band IG component values are unavailable/confidential for this selection.")


### Network-cost interpretation

The component data tell us **how much of the invoiced price is attributed to networks**, but not that every customer faces the same tariff structure.

CRU explains that network charges fund transmission/distribution operation and investment. Its 2025/26 tariff decision says network charges increased as Ireland continued investing in a more resilient, smarter and cleaner electricity grid.

We therefore treat network costs as a genuine long-run price driver, while avoiding a false one-to-one claim between aggregate CRU revenues and the price paid by a specific large user.


## 14. First forecasting benchmark

Before building a 2035 model we need a baseline that is genuinely hard to beat.

We use **rolling-origin one-step-ahead validation**:

- initial training data: 2008S1–2017S2
- validation: 2018S1–2025S2
- no random train/test split
- no future information allowed into a prediction

Models:

1. **Naive:** next semester = latest observed price
2. **Seasonal naive:** next semester = same semester one year earlier
3. **Linear trend**
4. **Holt damped trend**
5. **ARX Ridge:** lagged retail + lagged gas + semester + time

The ARX model uses only lagged gas values, so it does not peek at the future.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.holtwinters import Holt

forecast_df = prices.copy().reset_index(drop=True)
forecast_df["t"] = np.arange(len(forecast_df))

validation_start = forecast_df.index[
    forecast_df["period"].eq("2018S1")
][0]

model_names = [
    "Naive",
    "Seasonal Naive",
    "Linear Trend",
    "Holt Damped",
    "ARX Ridge",
]
predictions = {name: [] for name in model_names}
actual = []
validation_periods = []

for t in range(validation_start, len(forecast_df)):
    train = forecast_df.iloc[:t].copy()
    actual.append(forecast_df.loc[t, "electricity_ie_c_per_kwh"])
    validation_periods.append(forecast_df.loc[t, "period"])

    predictions["Naive"].append(
        train.iloc[-1]["electricity_ie_c_per_kwh"]
    )
    predictions["Seasonal Naive"].append(
        train.iloc[-2]["electricity_ie_c_per_kwh"]
    )

    linear = LinearRegression().fit(
        train[["t"]],
        train["electricity_ie_c_per_kwh"],
    )
    predictions["Linear Trend"].append(
        float(linear.predict(pd.DataFrame({"t": [t]}))[0])
    )

    holt = Holt(
        train["electricity_ie_c_per_kwh"].values,
        damped_trend=True,
        initialization_method="estimated",
    ).fit(optimized=True)
    predictions["Holt Damped"].append(
        float(holt.forecast(1)[0])
    )

    X_train, y_train = [], []
    for j in range(2, t):
        X_train.append([
            forecast_df.loc[j-1, "electricity_ie_c_per_kwh"],
            forecast_df.loc[j-2, "electricity_ie_c_per_kwh"],
            forecast_df.loc[j-1, "gas_ie_c_per_kwh"],
            forecast_df.loc[j-2, "gas_ie_c_per_kwh"],
            forecast_df.loc[j, "semester"],
            j,
        ])
        y_train.append(
            forecast_df.loc[j, "electricity_ie_c_per_kwh"]
        )

    arx = Ridge(alpha=1.0).fit(X_train, y_train)
    X_next = [[
        forecast_df.loc[t-1, "electricity_ie_c_per_kwh"],
        forecast_df.loc[t-2, "electricity_ie_c_per_kwh"],
        forecast_df.loc[t-1, "gas_ie_c_per_kwh"],
        forecast_df.loc[t-2, "gas_ie_c_per_kwh"],
        forecast_df.loc[t, "semester"],
        t,
    ]]
    predictions["ARX Ridge"].append(
        float(arx.predict(X_next)[0])
    )


In [ ]:
evaluation_rows = []

for model_name, pred in predictions.items():
    y_true = np.array(actual)
    y_pred = np.array(pred)

    evaluation_rows.append({
        "model": model_name,
        "MAE_c_per_kwh": mean_absolute_error(y_true, y_pred),
        "RMSE_c_per_kwh": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAPE_pct": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "R2": r2_score(y_true, y_pred),
        "validation_start": "2018S1",
        "validation_end": "2025S2",
        "observations": len(y_true),
    })

forecast_evaluation = (
    pd.DataFrame(evaluation_rows)
    .sort_values("RMSE_c_per_kwh")
    .reset_index(drop=True)
)

display(forecast_evaluation)
forecast_evaluation.to_csv(
    PROCESSED_DIR / "forecast_benchmark_evaluation.csv",
    index=False,
)

best_model = forecast_evaluation.iloc[0]
print(
    f"Best benchmark: {best_model['model']} | "
    f"RMSE {best_model['RMSE_c_per_kwh']:.2f} c/kWh | "
    f"MAPE {best_model['MAPE_pct']:.2f}%"
)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
plot_df = forecast_evaluation.sort_values("RMSE_c_per_kwh")
ax.barh(plot_df["model"], plot_df["RMSE_c_per_kwh"])
ax.invert_yaxis()
ax.set_title("Walk-forward forecast benchmark — 2018S1 to 2025S2")
ax.set_xlabel("RMSE (cent/kWh) — lower is better")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

prediction_long = []
for i, period in enumerate(validation_periods):
    for model_name in model_names:
        prediction_long.append({
            "period": period,
            "actual_c_per_kwh": actual[i],
            "model": model_name,
            "prediction_c_per_kwh": predictions[model_name][i],
        })

prediction_long = pd.DataFrame(prediction_long)
prediction_long.to_csv(
    PROCESSED_DIR / "forecast_benchmark_predictions.csv",
    index=False,
)


### Baseline result

On the current 2018S1–2025S2 walk-forward window, the **naive last-value forecast is the benchmark to beat**.

That is not a failure. It is evidence that the retail price series is difficult to forecast through structural shocks, and that adding predictors must earn its complexity through better out-of-sample performance.

The next model will add the wholesale-electricity signal carefully. Because the CSO Day-Ahead-based monthly index begins in 2021 in our downloaded series, that feature has a much shorter validation window and must be evaluated separately rather than mixed into the full 2008–2025 benchmark.


## 15. Next modelling stage

Before producing any 2026–2035 price path we still need:

1. a validated **wholesale-feature model** on the shorter 2021–2025 overlap;
2. price-component history from Eurostat/SEAI;
3. network / policy scenarios rather than a guessed trend;
4. scenario assumptions for wholesale/gas conditions;
5. uncertainty intervals.

Only after those tests will we produce a long-run forecast.


## 16. Extended SEMOpx wholesale feature

The first wholesale-lag diagnostic used CSO WPM37 from 2021 onward. The official SEMOpx look-back workbooks now let us extend the Day-Ahead Market signal back to market go-live in late 2018.

For a fair predictive test we:

1. aggregate hourly SEMOpx DAM prices to half-years;
2. normalise the complete 2019–2020 semesters to a 2021-average = 100 basis;
3. use CSO WPM37 from 2021 onward;
4. verify that the two official wholesale series overlap closely after normalisation;
5. test lagged wholesale features using walk-forward validation.

This creates a continuous wholesale feature without inventing missing 2025 DAM prices.


In [ ]:
from openpyxl import load_workbook
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEMOPX_OLD = RAW_DIR / "semopx_go_live_to_2020.xlsx"
SEMOPX_NEW = RAW_DIR / "semopx_2021_onwards.xlsx"

def load_semopx_dam(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb["auctions_to"]
    rows = ws.iter_rows(values_only=True)
    header = next(rows)
    idx = {name: i for i, name in enumerate(header)}
    records = []
    for row in rows:
        if row[idx["auction"]] == "DAM":
            records.append({
                "date": row[idx["DeliveryDate"]],
                "dam_eur_mwh": row[idx["price_eur"]],
            })
    out = pd.DataFrame(records)
    out["date"] = pd.to_datetime(out["date"])
    out["dam_eur_mwh"] = pd.to_numeric(out["dam_eur_mwh"], errors="coerce")
    return out.dropna().sort_values("date")

semopx_hourly = pd.concat(
    [load_semopx_dam(SEMOPX_OLD), load_semopx_dam(SEMOPX_NEW)],
    ignore_index=True,
).drop_duplicates()

semopx_hourly["year"] = semopx_hourly["date"].dt.year
semopx_hourly["semester"] = np.where(
    semopx_hourly["date"].dt.month <= 6, 1, 2
)

semopx_half = (
    semopx_hourly.groupby(["year", "semester"], as_index=False)
    .agg(
        dam_eur_mwh=("dam_eur_mwh", "mean"),
        hourly_observations=("dam_eur_mwh", "size"),
    )
)

display(semopx_half.tail(12))


In [ ]:
# Normalise SEMOpx DAM to 2021 average = 100 and compare with CSO WPM37.
dam_2021_base = semopx_half.loc[
    semopx_half["year"].eq(2021), "dam_eur_mwh"
].mean()

semopx_half["dam_index_2021_100"] = (
    semopx_half["dam_eur_mwh"] / dam_2021_base * 100
)

wpm_half = wholesale.copy()
wpm_half["year"] = wpm_half["date"].dt.year
wpm_half["semester"] = np.where(wpm_half["date"].dt.month <= 6, 1, 2)
wpm_half = (
    wpm_half.groupby(["year", "semester"], as_index=False)
    ["wholesale_electricity_index"].mean()
    .rename(columns={"wholesale_electricity_index": "wpm_index"})
)

overlap = semopx_half.merge(wpm_half, on=["year", "semester"], how="inner")
# Exclude incomplete SEMOpx semesters from validation.
overlap["complete_semester"] = overlap["hourly_observations"] > 4000
overlap_complete = overlap[overlap["complete_semester"]].copy()

overlap_corr = overlap_complete[
    "dam_index_2021_100"
].corr(overlap_complete["wpm_index"])
overlap_mae = (
    overlap_complete["dam_index_2021_100"]
    - overlap_complete["wpm_index"]
).abs().mean()

print(f"SEMOpx vs CSO overlap correlation: {overlap_corr:.4f}")
print(f"Mean absolute index difference: {overlap_mae:.3f} points")

display(
    overlap_complete[
        ["year", "semester", "dam_index_2021_100", "wpm_index"]
    ]
)

fig, ax = plt.subplots(figsize=(11, 5.5))
labels = (
    overlap_complete["year"].astype(str)
    + "S"
    + overlap_complete["semester"].astype(str)
)
ax.plot(labels, overlap_complete["dam_index_2021_100"], marker="o",
        label="SEMOpx DAM normalised")
ax.plot(labels, overlap_complete["wpm_index"], marker="o",
        label="CSO WPM37 electricity index")
ax.set_title("SEMOpx vs CSO wholesale signal overlap")
ax.set_ylabel("Index (2021 average = 100)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.show()


### Validation result

The normalised SEMOpx DAM series and CSO WPM37 signal track each other extremely closely across complete overlapping semesters.

That gives us a defensible way to extend the wholesale feature backward:

- **2019–2020:** normalised SEMOpx DAM;
- **2021 onward:** CSO WPM37.

We do not use the incomplete late-2024 SEMOpx semester to validate the overlap.


In [ ]:
# Build the continuous wholesale feature.
semopx_pre2021 = semopx_half[
    semopx_half["year"].between(2019, 2020)
    & semopx_half["hourly_observations"].gt(4000)
].copy()
semopx_pre2021["wholesale_index"] = semopx_pre2021["dam_index_2021_100"]

wpm_post2021 = wpm_half[
    wpm_half["year"].between(2021, 2025)
].copy()
wpm_post2021["wholesale_index"] = wpm_post2021["wpm_index"]

wholesale_feature = pd.concat(
    [
        semopx_pre2021[["year", "semester", "wholesale_index"]],
        wpm_post2021[["year", "semester", "wholesale_index"]],
    ],
    ignore_index=True,
).sort_values(["year", "semester"])

wholesale_feature["period"] = (
    wholesale_feature["year"].astype(str)
    + "S"
    + wholesale_feature["semester"].astype(str)
)

wholesale_feature.to_csv(
    PROCESSED_DIR / "wholesale_feature_halfyear_2019_2025.csv",
    index=False,
)

display(wholesale_feature)


In [ ]:
# Build lagged predictive features.
feature_frame = (
    prices[
        [
            "period", "year", "semester",
            "electricity_ie_c_per_kwh", "gas_ie_c_per_kwh"
        ]
    ]
    .merge(
        wholesale_feature[["period", "wholesale_index"]],
        on="period",
        how="inner",
    )
    .sort_values(["year", "semester"])
    .reset_index(drop=True)
)

feature_frame["retail_lag1"] = (
    feature_frame["electricity_ie_c_per_kwh"].shift(1)
)
feature_frame["gas_lag1"] = feature_frame["gas_ie_c_per_kwh"].shift(1)
feature_frame["wholesale_lag1"] = feature_frame["wholesale_index"].shift(1)

# Use 2022S1–2025S2 so every model has a reasonable initial training sample.
test_periods = feature_frame.loc[
    feature_frame["year"].between(2022, 2025),
    "period",
].tolist()

model_specs = {
    "Naive last retail": None,
    "AR(1) linear": ["retail_lag1"],
    "Wholesale lag linear": ["wholesale_lag1"],
    "Ridge retail + wholesale lag": [
        "retail_lag1", "wholesale_lag1"
    ],
    "Ridge retail + wholesale + gas lag": [
        "retail_lag1", "wholesale_lag1", "gas_lag1"
    ],
}

prediction_rows = []

for period in test_periods:
    test_idx = feature_frame.index[
        feature_frame["period"].eq(period)
    ][0]

    train = feature_frame.loc[: test_idx - 1].dropna(
        subset=[
            "electricity_ie_c_per_kwh",
            "retail_lag1",
            "wholesale_lag1",
            "gas_lag1",
        ]
    )
    test = feature_frame.loc[[test_idx]]
    actual = float(test["electricity_ie_c_per_kwh"].iloc[0])

    for model_name, features in model_specs.items():
        if model_name == "Naive last retail":
            forecast = float(test["retail_lag1"].iloc[0])
        else:
            X_train = train[features]
            y_train = train["electricity_ie_c_per_kwh"]
            X_test = test[features]

            if model_name.startswith("Ridge"):
                model = make_pipeline(
                    StandardScaler(),
                    Ridge(alpha=1.0),
                )
            else:
                model = LinearRegression()

            model.fit(X_train, y_train)
            forecast = float(model.predict(X_test)[0])

        prediction_rows.append({
            "period": period,
            "model": model_name,
            "actual_c_per_kwh": actual,
            "forecast_c_per_kwh": forecast,
        })

wholesale_predictions = pd.DataFrame(prediction_rows)

evaluation_rows = []
for model_name, group in wholesale_predictions.groupby("model"):
    actual = group["actual_c_per_kwh"]
    forecast = group["forecast_c_per_kwh"]
    evaluation_rows.append({
        "model": model_name,
        "MAE": mean_absolute_error(actual, forecast),
        "RMSE": mean_squared_error(actual, forecast) ** 0.5,
        "MAPE": (
            np.abs((actual - forecast) / actual).mean() * 100
        ),
        "R2": r2_score(actual, forecast),
    })

wholesale_evaluation = (
    pd.DataFrame(evaluation_rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(wholesale_evaluation)

wholesale_predictions.to_csv(
    PROCESSED_DIR / "wholesale_feature_model_predictions.csv",
    index=False,
)
wholesale_evaluation.to_csv(
    PROCESSED_DIR / "wholesale_feature_model_evaluation.csv",
    index=False,
)


In [ ]:
# Compare the persistence baseline with the best exogenous-feature model.
chart_data = (
    wholesale_predictions[
        wholesale_predictions["model"].isin(
            [
                "Naive last retail",
                "Ridge retail + wholesale + gas lag",
            ]
        )
    ]
    .pivot(
        index="period",
        columns="model",
        values="forecast_c_per_kwh",
    )
)

actual = (
    wholesale_predictions.drop_duplicates("period")
    .set_index("period")["actual_c_per_kwh"]
)

chart_data = chart_data.join(actual)

fig, ax = plt.subplots(figsize=(11, 5.8))
ax.plot(chart_data.index, chart_data["actual_c_per_kwh"],
        marker="o", linewidth=2.5, label="Actual retail price")
ax.plot(chart_data.index, chart_data["Naive last retail"],
        marker="o", linewidth=2, label="Naive forecast")
ax.plot(
    chart_data.index,
    chart_data["Ridge retail + wholesale + gas lag"],
    marker="o",
    linewidth=2,
    label="Retail + wholesale + gas lag",
)
ax.set_title("Wholesale-feature model validation, 2022S1–2025S2")
ax.set_ylabel("cent/kWh")
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.show()


## 17. What the wholesale-feature test tells us

The relationship is economically meaningful, but **predictive usefulness is a harder test than correlation**.

On the common 2022S1–2025S2 walk-forward window:

- the **naive last-price model remains the strongest**;
- adding lagged wholesale prices does **not** reduce RMSE;
- adding both wholesale and gas narrows the gap, but still does not beat persistence.

This is an important result, not a failure.

It means we should keep wholesale and gas as:

1. explanatory variables for understanding the 2021–2023 price shock;
2. explicit inputs in future **scenario modelling**;

but we should not claim they are proven superior point-forecast predictors with the current sample.

The project therefore moves from “find a clever algorithm” to a more defensible long-horizon approach:

> **baseline statistical forecast + explicit wholesale/gas/network scenarios + uncertainty ranges.**


## 18. Next stage — 2026–2035 scenario engine

The long-run model will not output one supposedly certain 2035 price.

We will build three transparent paths:

### Low-cost transition
- lower wholesale/gas pressure;
- high renewable penetration;
- stronger interconnection/flexibility;
- controlled network-cost growth.

### Central
- moderate wholesale/gas assumptions;
- planned network investment;
- continued electricity-demand growth.

### High-cost / constrained
- sustained fuel-price pressure;
- stronger demand;
- higher network-cost recovery;
- slower system flexibility / greater constraint exposure.

Each scenario will produce a **price range**, not a single-point prophecy.

Before fitting those scenarios we will add the demand variables from Project 01 so the final model can test demand × price together.


## 19. 2026–2035 electricity-price scenario engine

The forecasting tests above give us an important result: a more complex statistical model does **not** consistently outperform the simple persistence baseline.

For a 10-year horizon, the defensible next step is therefore **scenario analysis**, not an overconfident point forecast.

We model the retail price as three broad components:

1. **Energy & supply**
2. **Network**
3. **Taxes & levies**

The component weights are calibrated from Eurostat's 2025 annual non-household electricity-price component data, but normalised to the SEAI target-price anchors. This avoids directly mixing Eurostat annual component totals with the SEAI biannual target series.

Policy context:
- Ireland targets **80% renewable electricity by 2030**.
- CRU Price Review 6 approves **€13.8bn baseline electricity-network investment for 2026–2030**, with scope to rise to **€18.9bn**.
- CRU has stated that this investment will increase network charges in the near term while enabling the low-carbon transition.

These policy facts inform scenario direction. **They are not official electricity-price forecasts.**


In [ ]:
# 2025 target-price anchors
WA_BASE_2025 = float(
    prices.query("year == 2025")["electricity_ie_c_per_kwh"].mean()
)
IG_BASE_2025S2 = float(
    bands_2025s2.query("band == 'IG'")["price_c_per_kwh"].iloc[0]
)

print(f"Weighted-average 2025 anchor: {WA_BASE_2025:.3f} c/kWh ex VAT")
print(f"Band IG 2025S2 anchor: {IG_BASE_2025S2:.1f} c/kWh ex VAT")


### Component-share calibration

Eurostat's 2025 annual component table gives us relative component structure.

For the weighted business series we use the `TOT_KWH` component proportions.  
For the large-user scenario we use `MWH_GE150000` (Band IG).

Only three mutually exclusive top-level components are used:

- `NRG_SUP` — energy & supply
- `NETC` — network
- `TAX_FEE_LEV_CHRG` — taxes, fees, levies and charges

The shares are then normalised to the SEAI ex-VAT price anchors.


In [ ]:
components_raw = pd.read_csv(RAW_DIR / "eurostat_nrg_pc_205_c_ireland_2017_2025.csv")

def component_value(nrg_prc, nrg_cons, year="2025"):
    row = components_raw[
        (components_raw["nrg_prc"] == nrg_prc)
        & (components_raw["nrg_cons"] == nrg_cons)
    ]
    if row.empty:
        raise ValueError(f"Missing component: {nrg_prc}, {nrg_cons}")
    return float(row[year].iloc[0]) * 100  # EUR/kWh -> cent/kWh

def calibrated_component_shares(nrg_cons):
    raw = {
        "energy_supply": component_value("NRG_SUP", nrg_cons),
        "network": component_value("NETC", nrg_cons),
        "taxes_levies": component_value("TAX_FEE_LEV_CHRG", nrg_cons),
    }
    total = sum(raw.values())
    return pd.Series({k: v / total for k, v in raw.items()})

wa_shares = calibrated_component_shares("TOT_KWH")
ig_shares = calibrated_component_shares("MWH_GE150000")

share_table = pd.DataFrame({
    "Weighted-average business": wa_shares,
    "Band IG": ig_shares,
})
display((share_table * 100).round(1).rename_axis("component"))


## 20. Scenario assumptions

These rates are **analytical assumptions**, not official forecasts.

The key design principle is transparency: every assumption is visible and editable.

All component-growth assumptions below are **real** changes, before general inflation.


In [ ]:
scenario_assumptions = pd.DataFrame([
    {
        "scenario": "Low-cost transition",
        "energy_2026": -0.02,
        "energy_2027_2030": -0.025,
        "energy_2031_2035": -0.01,
        "network_2026_2030": 0.01,
        "network_2031_2035": 0.00,
        "levies_2026_2030": -0.01,
        "levies_2031_2035": 0.00,
    },
    {
        "scenario": "Central",
        "energy_2026": 0.02,
        "energy_2027_2030": -0.01,
        "energy_2031_2035": 0.00,
        "network_2026_2030": 0.03,
        "network_2031_2035": 0.01,
        "levies_2026_2030": 0.00,
        "levies_2031_2035": 0.005,
    },
    {
        "scenario": "High-cost / constrained",
        "energy_2026": 0.08,
        "energy_2027_2030": 0.02,
        "energy_2031_2035": 0.015,
        "network_2026_2030": 0.05,
        "network_2031_2035": 0.02,
        "levies_2026_2030": 0.015,
        "levies_2031_2035": 0.01,
    },
])

display(scenario_assumptions)
scenario_assumptions.to_csv(
    PROCESSED_DIR / "price_scenario_assumptions_2026_2035.csv",
    index=False,
)


### Nominal-price translation

The core scenario is kept in **2025-price terms**.

For a second, more intuitive view we translate the real paths into nominal cents/kWh using:

- 2026: 3.0%
- 2027: 2.5%
- 2028: 2.1%
- 2029 onward: 2.0%

The 2026–2028 values follow the September 2026 ECB staff inflation path. From 2029 onward we use the ECB's 2% medium-term inflation target as a neutral translation assumption.

This inflation layer is **not** part of the electricity-system forecast.


In [ ]:
inflation_assumption = {
    2026: 0.030,
    2027: 0.025,
    2028: 0.021,
    **{year: 0.020 for year in range(2029, 2036)},
}

pd.DataFrame(
    [{"year": y, "inflation_assumption": r} for y, r in inflation_assumption.items()]
)


In [ ]:
def build_price_scenarios(base_price, shares, customer):
    base_components = {
        "energy_supply": base_price * shares["energy_supply"],
        "network": base_price * shares["network"],
        "taxes_levies": base_price * shares["taxes_levies"],
    }

    rows = []

    for _, a in scenario_assumptions.iterrows():
        c = base_components.copy()
        cumulative_inflation = 1.0

        rows.append({
            "customer": customer,
            "scenario": a["scenario"],
            "year": 2025,
            **c,
            "real_total_c_per_kwh": sum(c.values()),
            "cumulative_inflation_factor": cumulative_inflation,
            "nominal_total_c_per_kwh": sum(c.values()),
        })

        for year in range(2026, 2036):
            if year == 2026:
                c["energy_supply"] *= 1 + a["energy_2026"]
            elif year <= 2030:
                c["energy_supply"] *= 1 + a["energy_2027_2030"]
            else:
                c["energy_supply"] *= 1 + a["energy_2031_2035"]

            c["network"] *= 1 + (
                a["network_2026_2030"]
                if year <= 2030
                else a["network_2031_2035"]
            )

            c["taxes_levies"] *= 1 + (
                a["levies_2026_2030"]
                if year <= 2030
                else a["levies_2031_2035"]
            )

            cumulative_inflation *= 1 + inflation_assumption[year]

            rows.append({
                "customer": customer,
                "scenario": a["scenario"],
                "year": year,
                **c,
                "real_total_c_per_kwh": sum(c.values()),
                "cumulative_inflation_factor": cumulative_inflation,
                "nominal_total_c_per_kwh": sum(c.values()) * cumulative_inflation,
            })

    return pd.DataFrame(rows)

weighted_scenarios = build_price_scenarios(
    WA_BASE_2025, wa_shares, "Weighted-average business"
)
ig_scenarios = build_price_scenarios(
    IG_BASE_2025S2, ig_shares, "Band IG large user"
)

price_scenarios = pd.concat(
    [weighted_scenarios, ig_scenarios],
    ignore_index=True,
)

price_scenarios.to_csv(
    PROCESSED_DIR / "business_electricity_price_scenarios_2025_2035.csv",
    index=False,
)

display(
    price_scenarios[
        price_scenarios["year"].isin([2025, 2026, 2030, 2035])
    ][
        [
            "customer",
            "scenario",
            "year",
            "real_total_c_per_kwh",
            "nominal_total_c_per_kwh",
        ]
    ].round(2)
)


## 21. Weighted-average business price scenarios

The real-price chart answers:

> **How might electricity-system costs evolve if general inflation is stripped out?**

The nominal chart answers:

> **What could those same scenario paths look like in the cents/kWh actually quoted in future years?**


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for scenario, group in weighted_scenarios.groupby("scenario"):
    ax.plot(
        group["year"],
        group["real_total_c_per_kwh"],
        marker="o",
        linewidth=2.4,
        label=scenario,
    )

ax.set_title("Irish business electricity price scenarios — real 2025 cents")
ax.set_ylabel("cent/kWh, ex VAT (2025-price terms)")
ax.set_xlabel("Year")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for scenario, group in weighted_scenarios.groupby("scenario"):
    ax.plot(
        group["year"],
        group["nominal_total_c_per_kwh"],
        marker="o",
        linewidth=2.4,
        label=scenario,
    )

ax.set_title("Irish business electricity price scenarios — nominal translation")
ax.set_ylabel("cent/kWh, ex VAT (nominal)")
ax.set_xlabel("Year")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()


### 2035 weighted-average business scenario range

Using the current analytical assumptions:

- **Low-cost transition:** about **21.2 c/kWh in real 2025 terms**
- **Central:** about **24.4 c/kWh**
- **High-cost / constrained:** about **30.1 c/kWh**

After the separate inflation translation, those become roughly:

- **26.2 c/kWh nominal**
- **30.2 c/kWh nominal**
- **37.2 c/kWh nominal**

This is a **scenario envelope**, not a statistical confidence interval.


## 22. Large-user Band IG scenario

Band IG covers non-household customers using more than **150,000 MWh/year**.

It is not a data-centre tariff, but it is the best public large-consumer price benchmark available for the later Demand × Price project.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for scenario, group in ig_scenarios.groupby("scenario"):
    ax.plot(
        group["year"],
        group["real_total_c_per_kwh"],
        marker="o",
        linewidth=2.4,
        label=scenario,
    )

ax.set_title("Large-user Band IG electricity price scenarios")
ax.set_ylabel("cent/kWh, ex VAT (2025-price terms)")
ax.set_xlabel("Year")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()

display(
    ig_scenarios[
        ig_scenarios["year"].isin([2025, 2030, 2035])
    ][
        [
            "scenario",
            "year",
            "real_total_c_per_kwh",
            "nominal_total_c_per_kwh",
        ]
    ].round(2)
)


### 2035 Band IG scenario range

In real 2025-price terms:

- **Low-cost transition:** ~14.1 c/kWh
- **Central:** ~16.2 c/kWh
- **High-cost / constrained:** ~20.3 c/kWh

This is the price series we can later combine with Project 01's data-centre demand scenarios without pretending that data centres pay the weighted-average business tariff.


## 23. What the scenario engine means

The historical modelling stage told us something important:

- retail prices are persistent;
- wholesale and gas prices matter for explaining shocks;
- lagged wholesale features alone do not reliably beat a simple out-of-sample persistence forecast;
- network costs are becoming structurally important because Ireland is entering an unusually large investment cycle.

So the 2026–2035 output is intentionally a **transparent scenario model**.

### What would change the result?

The user can change:

- energy/supply growth assumptions;
- network-cost growth;
- taxes/levies;
- inflation translation.

This makes the model suitable for the next project, where we will connect:

**electricity demand → system investment → wholesale conditions → large-user and business price scenarios.**
